# Phase 1 — Document Preprocessing

This notebook prepares the document corpus for the Visual-Doc Assistant.

**Workflow:** PDF → page images → local project data → ChromaDB initialization.

Run this notebook before Phase 2. Generated data and database files are intentionally excluded from GitHub via `.gitignore`.

In [ ]:
5# Install the system-level dependency for reading PDFs
!apt-get update
!apt-get install -y poppler-utils

# Install the required Python libraries
!pip install pdf2image chromadb

In [ ]:
import os
from pdf2image import convert_from_path

def process_pdf_to_images(pdf_path, output_folder):
    # Create output directory if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Convert PDF pages to a list of PIL Image objects
    # 300 DPI is usually the standard for good OCR/Vision results
    pages = convert_from_path(pdf_path, 300)

    image_paths = []
    for i, page in enumerate(pages):
        image_name = f"page_{i+1}.png"
        image_path = os.path.join(output_folder, image_name)
        page.save(image_path, "PNG")
        image_paths.append(image_path)
        print(f"Saved: {image_path}")

    return image_paths

# Usage
pdf_file = "./2407.01449v6.pdf"

output_folder = "./processed_images"
images = process_pdf_to_images(pdf_file, output_folder)

In [ ]:
import chromadb
from chromadb.config import Settings

def setup_vector_db(db_path):
    # Initialize the Persistent Client (stores data on disk)
    client = chromadb.PersistentClient(path=db_path)

    # Create or Get a collection for the Visual-Doc Assistant
    # We use 'cosine' space for similarity matching in later phases
    collection = client.get_or_create_collection(
        name="visual_doc_collection",
        metadata={"hnsw:space": "cosine"}
    )

    print(f"Collection '{collection.name}' is ready.")
    return collection

# Usage
db_path = "./chroma_db_storage"
my_collection = setup_vector_db(db_path)

In [ ]:
print(my_collection.name)
print(my_collection.count())